In [1]:
import os
import sys
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

import smoothie
smoothie.suppress_warnings()

In [2]:
def pcc_cutoff_for_top_x_percent(pearsonR_mat, top_x_percent):
    upper_tri = pearsonR_mat[np.triu_indices_from(pearsonR_mat, k=1)]
    cutoff = np.percentile(upper_tri, 100 - top_x_percent)
    return cutoff

In [3]:
pearsonR_mat_mouse = np.load("../data/mouse_E16.5_E1S1_pearsonR.npy")
gene_names_mouse = np.load("../data/mouse_E16.5_E1S1_gene_names.npy", allow_pickle=True)

pearsonR_mat_human = np.load("../data/GSM9629357_CS23_E2S1_bin1_pearsonR.npy")
gene_names_human = np.load("../data/GSM9629357_CS23_E2S1_bin1_gene_names.npy", allow_pickle=True)

In [4]:
edge_list_mouse, node_label_df_mouse = smoothie.make_spatial_network(
    pearsonR_mat=pearsonR_mat_mouse, # don't change
    gene_names=gene_names_mouse, # don't change
    pcc_cutoff=float(pcc_cutoff_for_top_x_percent(pearsonR_mat_mouse, 0.01)),
    clustering_power=3,
    #output_folder='../Cytoscape'
)

# Filter node_label_df for only genes within modules of size 2 or more.
modules_df_mouse = node_label_df_mouse.groupby('module_label').filter(lambda x: len(x) >= 2)

# Examine modules
modules_df_mouse

,name,module_label,degree,weighted_degree,rescaled_weighted_degree,clustering_coeff,margin_score
0,Tuba1a,1,294,216.103867,16.464734,0.194446,0.902409
1,Rtn1,1,276,201.808817,12.615974,0.209750,0.978963
2,Crmp1,1,249,181.386469,11.123636,0.256089,0.996818
3,Tubb2b,1,250,181.619752,10.946271,0.245462,0.969822
4,Stmn2,1,236,171.883966,10.424670,0.272449,0.999678
...,...,...,...,...,...,...,...
1419,Mtf1,146,1,0.654376,0.000006,0.000000,1.000000
1420,Ndufb8,147,4,2.834163,0.192103,0.500000,0.999397
1421,Gm20538,147,1,0.850917,0.191756,0.000000,1.000000
1422,Nefl,148,2,1.587327,0.561314,0.000000,1.000000


In [ ]:
edge_list_human, node_label_df_human = smoothie.make_spatial_network(
    pearsonR_mat=pearsonR_mat_human, # don't change
    gene_names=gene_names_human, # don't change
    pcc_cutoff=float(pcc_cutoff_for_top_x_percent(pearsonR_mat_human, 0.01)),
    clustering_power=3,
    #output_folder='../Cytoscape'
)

# Filter node_label_df for only genes within modules of size 2 or more.
modules_df_human = node_label_df_human.groupby('module_label').filter(lambda x: len(x) >= 2)

# Examine modules
modules_df_human

,name,module_label,degree,weighted_degree,rescaled_weighted_degree,clustering_coeff,margin_score
0,RPL13A,1,433,213.981035,3.580172e+01,0.348826,0.786747
1,EEF1A1,1,434,214.647108,3.482404e+01,0.347123,0.797070
2,RPS8,1,405,199.916936,3.175550e+01,0.392324,0.833424
3,RPS23,1,422,205.674173,3.139500e+01,0.363995,0.849298
4,RPL37A,1,380,191.533258,3.112638e+01,0.437523,0.745258
...,...,...,...,...,...,...,...
1450,NDUFB8,131,1,0.573558,5.597477e-02,0.000000,1.000000
1451,AC142391.1,132,1,0.403835,2.558731e-03,0.000000,1.000000
1452,ECSCR,132,1,0.403835,2.558731e-03,0.000000,1.000000
1453,AC010624.2,133,1,0.310827,9.309333e-09,0.000000,1.000000


In [30]:
def find_lowercase_orthologs(modules_df_a, modules_df_b, name_a="species_a", name_b="species_b"):
    """
    Preliminary ortholog search: genes are matched by lowercasing both name columns.
    Returns a DataFrame of shared gene names with their module labels from each species.
    """
    df_a = modules_df_a[["name", "module_label"]].copy()
    df_b = modules_df_b[["name", "module_label"]].copy()

    df_a["name_lower"] = df_a["name"].str.lower()
    df_b["name_lower"] = df_b["name"].str.lower()

    merged = df_a.merge(df_b, on="name_lower", suffixes=(f"_{name_a}", f"_{name_b}"))
    merged = merged.rename(columns={"name_lower": "name_lower_shared"})

    return merged.reset_index(drop=True)


orthologs_df = find_lowercase_orthologs(modules_df_mouse, modules_df_human, name_a="mouse", name_b="human")
print(f"Found {len(orthologs_df)} shared gene names (case-insensitive)")
orthologs_df

Found 562 shared gene names (case-insensitive)


,name_mouse,module_label_mouse,name_lower_shared,name_human,module_label_human
0,Tuba1a,1,tuba1a,TUBA1A,3
1,Rtn1,1,rtn1,RTN1,3
2,Crmp1,1,crmp1,CRMP1,3
3,Tubb2b,1,tubb2b,TUBB2B,3
4,Stmn2,1,stmn2,STMN2,3
...,...,...,...,...,...
557,Ndufb9,139,ndufb9,NDUFB9,1
558,Bloc1s1,142,bloc1s1,BLOC1S1,85
559,Ndufb8,147,ndufb8,NDUFB8,131
560,Nefl,148,nefl,NEFL,13


In [25]:
gene = "SEPT7-AS1"

gene.lower() in orthologs_df["name_lower_shared"].values


False